# NB12 · Exceptions

`intermediate/` · notebook 12 of 32 · 36 questions + a mini-project

An exception is Python's way of saying "I can't carry on from here". You've
already met plenty: `ValueError`, `KeyError`, `ZeroDivisionError`, and
`RecursionError` in NB11. NB07 showed a first `try`/`except`. This notebook
goes deep: catching the right errors, raising your own, chaining them, and
reporting several at once.

Exceptions are classes arranged in a family tree. `KeyError` and
`IndexError` are both kinds of `LookupError`, which is a kind of
`Exception`, which is a kind of `BaseException`. An `except` clause catches
the class you name **and everything below it**, which is why the order of
your `except` clauses matters.

Classes with methods come in NB14 and `with` comes in NB13, so the custom
exceptions here use the one-line form `class MyError(Exception): pass`, and
clean-up code goes in `finally`.

**About Python 3.14:** two things changed. You may now write
`except ValueError, TypeError:` without brackets (when there's no `as`), and a
`return` inside `finally` now shows a `SyntaxWarning`. Questions that
depend on this are marked.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB11: functions and recursion

## What you'll practise

- The built-in exception hierarchy (LookupError above KeyError and IndexError)
- try / except / else / finally; several except clauses; as e
- raise, re-raise, raise ... from (exception chaining)
- Custom exceptions (the one-line class form; full classes come in NB14)
- EAFP vs LBYL
- assert, and why it isn't input validation
- ExceptionGroup and except* (3.11+), add_note()
- e.args, __cause__ vs __context__, the traceback module
- 3.14: except A, B without brackets; return in finally now warns

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB12-Q01 · L1 · try / except · Write

Print `total / people`. If `people` is zero, print `cannot divide by zero`
instead of crashing.

**Starting code (already in the cell below):**

```python
total = 120
people = 0
```

<details><summary><b>Hint</b></summary>

Put the division inside `try:`, and catch the error it raises with `except ZeroDivisionError:`.

</details>

<details><summary><b>Expected output</b></summary>

```text
cannot divide by zero
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    print(total / people)
except ZeroDivisionError:
    print("cannot divide by zero")
```

Python runs the `try` block. When `total / people` raises `ZeroDivisionError`, it jumps straight to the matching `except` block and the program carries on. If no error happens, the `except` block is skipped.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    share = total / people
except ZeroDivisionError:
    print("cannot divide by zero")
else:
    print(share)
```

Seniors keep **only** the risky line inside `try`. The `else` block (Q06) runs when nothing went wrong, so a bug in the printing code can't be mistaken for a division problem.

</details>

In [ ]:
total = 120
people = 0

# your code here


### NB12-Q02 · L1 · Built-in Exceptions · Error

Which error does this raise?

**Which error does this raise?**

```python
stock = {"apple": 12, "banana": 5}
print(stock["pear"])
```

<details><summary><b>Hint</b></summary>

Looking up a dict key that isn't there.

</details>

<details><summary><b>Expected output</b></summary>

```text
KeyError
```

</details>

<details><summary><b>Solution</b></summary>

`KeyError`: the dict has no `"pear"` key. Lists raise `IndexError` for a position that doesn't exist; dicts raise `KeyError` for a key that doesn't exist. Both are kinds of `LookupError` (see Q10).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
stock = {"apple": 12, "banana": 5}
print(stock.get("pear", 0))
```

When a missing key has a sensible default, `dict.get()` (NB06) avoids the exception entirely. Use `try`/`except KeyError` when a missing key is genuinely unusual and needs different handling.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB12-Q03 · L1 · try / except · Write

`text` came from a form. Print `Quantity: ` followed by the number, or, if
it isn't a whole number, print `not a number: '12a'` (the text in quotes).

**Starting code (already in the cell below):**

```python
text = "12a"
```

<details><summary><b>Hint</b></summary>

`int(text)` raises `ValueError` for text like this. `{text!r}` in an f-string shows the text with quotes around it.

</details>

<details><summary><b>Expected output</b></summary>

```text
not a number: '12a'
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    print("Quantity:", int(text))
except ValueError:
    print(f"not a number: {text!r}")
```

`int("12a")` raises `ValueError` because the text has the right type (a string) but the wrong value. The `except ValueError` block catches exactly that error. `!r` uses `repr()`, which adds the quotes, so an empty or space-only input is still visible in the message.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    quantity = int(text)
except ValueError:
    print(f"not a number: {text!r}")
else:
    print("Quantity:", quantity)
```

Checking first with `text.isdigit()` looks tempting, but it rejects `"-3"` and `" 12 "`, which `int()` accepts. Trying the conversion and catching the failure is simpler and always agrees with `int()` itself (Q18).

</details>

In [ ]:
text = "12a"

# your code here


### NB12-Q04 · L1 · as e · Write

Call `check_qty(-2)`. Catch the error and print its message after
`Error: `:

```text
Error: quantity cannot be negative
```

**Starting code (already in the cell below):**

```python
def check_qty(qty):
    if qty < 0:
        raise ValueError("quantity cannot be negative")
    return qty
```

<details><summary><b>Hint</b></summary>

`except ValueError as e:` gives the exception object a name. Printing it shows its message.

</details>

<details><summary><b>Expected output</b></summary>

```text
Error: quantity cannot be negative
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    check_qty(-2)
except ValueError as e:
    print("Error:", e)
```

`raise ValueError("...")` creates an exception object carrying a message and throws it. `as e` catches that object and names it. `print(e)` shows the message, because `str()` of an exception is its message.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    check_qty(-2)
except ValueError as err:
    print(f"Error: {err}")
```

Python deletes the `as` name when the `except` block ends, so `err` can't be used afterwards. If you need the error later, copy it to another name inside the block.

</details>

In [ ]:
def check_qty(qty):
    if qty < 0:
        raise ValueError("quantity cannot be negative")
    return qty

# your code here


### NB12-Q05 · L1 · try / except · Predict

What runs and what is skipped? Predict the output.

**What does this print?**

```python
try:
    print("A")
    print(10 / 0)
    print("B")
except ZeroDivisionError:
    print("C")
print("D")
```

<details><summary><b>Hint</b></summary>

When a line in `try` raises, Python leaves the `try` block immediately. Does it ever come back?

</details>

<details><summary><b>Expected output</b></summary>

```text
A
C
D
```

</details>

<details><summary><b>Solution</b></summary>

`A` is printed. Then `10 / 0` raises, and Python jumps straight to the `except` block, so `B` is **never** printed: the rest of the `try` block is abandoned. `C` is printed by the handler, and then the program carries on after the whole statement with `D`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is why seniors keep `try` blocks short. Every line after the risky one is a line that silently doesn't run when something goes wrong.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB12-Q06 · L2 · else / finally · Predict

All four parts of a `try` statement, run twice. Predict the output.

**What does this print?**

```python
def convert(text):
    try:
        number = int(text)
    except ValueError:
        print("except: bad input")
    else:
        print("else: got", number)
    finally:
        print("finally: done with", text)

convert("42")
convert("4x")
```

<details><summary><b>Hint</b></summary>

`else` runs only when `try` raised nothing. `finally` runs every time, error or not.

</details>

<details><summary><b>Expected output</b></summary>

```text
else: got 42
finally: done with 42
except: bad input
finally: done with 4x
```

</details>

<details><summary><b>Solution</b></summary>

For `"42"`, nothing goes wrong: `except` is skipped, `else` runs, then `finally`. For `"4x"`, `int()` raises: `except` runs, `else` is skipped, then `finally` runs. The order is always try, then except **or** else, then finally.

</details>

<details><summary><b>Senior dev solution</b></summary>

Use `else` for code that should only run on success but shouldn't be protected by the `except`. Use `finally` for clean-up that must happen no matter what, such as closing a file or releasing a lock. With files, `with` (NB13) does that clean-up for you.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB12-Q07 · L2 · try / except · Write

Convert every string in `raw` to an `int`, skipping any that fail. Print
the list of numbers, then how many were skipped.

```text
[10, 7, 42]
skipped: 3
```

**Starting code (already in the cell below):**

```python
raw = ["10", "x", "7", "", "3.5", "42"]
```

<details><summary><b>Hint</b></summary>

Loop over `raw` with `try`/`except ValueError` inside the loop, so one bad value doesn't stop the rest. Count the failures in the `except` block.

</details>

<details><summary><b>Expected output</b></summary>

```text
[10, 7, 42]
skipped: 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
numbers = []
skipped = 0
for text in raw:
    try:
        numbers.append(int(text))
    except ValueError:
        skipped += 1
print(numbers)
print("skipped:", skipped)
```

The `try` is **inside** the loop, so a failure only affects that one item and the loop moves on. `""` and `"3.5"` fail too: `int()` accepts neither empty text nor a decimal point. If the `try` were around the whole loop, the first bad value would end the loop.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def to_int(text):
    try:
        return int(text)
    except ValueError:
        return None

parsed = [to_int(text) for text in raw]
numbers = [n for n in parsed if n is not None]
print(numbers)
print("skipped:", len(parsed) - len(numbers))
```

A comprehension can't contain `try`, so seniors move the risky part into a small helper that returns `None` on failure, then filter. Each piece is tiny and testable.

</details>

In [ ]:
raw = ["10", "x", "7", "", "3.5", "42"]

# your code here


### NB12-Q08 · L2 · raise · Write

Write `set_temperature(celsius)`. If `celsius` is below `-273.15`, it should
`raise ValueError("below absolute zero")`. Otherwise it returns
`"thermostat set to <celsius>"`. Call it for each reading and print the
result, or `error: ` plus the message:

```text
thermostat set to 21
error: below absolute zero
```

**Starting code (already in the cell below):**

```python
readings = [21, -300]
```

<details><summary><b>Hint</b></summary>

`raise` stops the function at once and sends the exception to the caller. The caller's `try`/`except ValueError as e` decides what to print.

</details>

<details><summary><b>Expected output</b></summary>

```text
thermostat set to 21
error: below absolute zero
```

</details>

<details><summary><b>Solution</b></summary>

```python
def set_temperature(celsius):
    if celsius < -273.15:
        raise ValueError("below absolute zero")
    return f"thermostat set to {celsius}"

for reading in readings:
    try:
        print(set_temperature(reading))
    except ValueError as e:
        print("error:", e)
```

The function's job is to *detect* a bad value; the caller's job is to *decide what to do* about it. `raise` separates the two. `ValueError` is the right class here: the argument is the right type (a number) with an impossible value.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
ABSOLUTE_ZERO_C = -273.15

def set_temperature(celsius):
    if celsius < ABSOLUTE_ZERO_C:
        raise ValueError("below absolute zero")
    return f"thermostat set to {celsius}"

for reading in readings:
    try:
        print(set_temperature(reading))
    except ValueError as e:
        print("error:", e)
```

Naming the limit as a constant documents it, and it's the one place to change if the unit ever changes. Raising beats returning a special value like `-1` or `None`: a caller can't forget to check an exception.

</details>

In [ ]:
readings = [21, -300]

# your code here


### NB12-Q09 · L2 · Several except Clauses · Write

`sales` maps each fruit to a list of daily sales. For each `(fruit, day)`
request, print the sale, or a message that says which part was wrong:

```text
apple day 1: 9
no such fruit: kiwi
no such day for pear: 5
```

**Starting code (already in the cell below):**

```python
sales = {"apple": [12, 9, 4], "pear": [3]}
requests = [("apple", 1), ("kiwi", 0), ("pear", 5)]
```

<details><summary><b>Hint</b></summary>

`sales[fruit][day]` can fail in two ways: a missing key or a missing position. Give each its own `except` clause.

</details>

<details><summary><b>Expected output</b></summary>

```text
apple day 1: 9
no such fruit: kiwi
no such day for pear: 5
```

</details>

<details><summary><b>Solution</b></summary>

```python
for fruit, day in requests:
    try:
        sold = sales[fruit][day]
    except KeyError:
        print("no such fruit:", fruit)
    except IndexError:
        print(f"no such day for {fruit}: {day}")
    else:
        print(f"{fruit} day {day}: {sold}")
```

Python checks the `except` clauses from top to bottom and runs the first one that matches. A missing fruit raises `KeyError` (from the dict); a missing day raises `IndexError` (from the list). Separate clauses let you react differently to each.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def describe(fruit, day):
    try:
        sold = sales[fruit][day]
    except KeyError:
        return f"no such fruit: {fruit}"
    except IndexError:
        return f"no such day for {fruit}: {day}"
    return f"{fruit} day {day}: {sold}"

for fruit, day in requests:
    print(describe(fruit, day))
```

Moving the logic into a function that *returns* the message makes it testable without capturing printed output, and each clause can `return` directly instead of needing `else`.

</details>

In [ ]:
sales = {"apple": [12, 9, 4], "pear": [3]}
requests = [("apple", 1), ("kiwi", 0), ("pear", 5)]

# your code here


### NB12-Q10 · L2 · Exception Hierarchy · Predict

One `except` clause, two different errors. Predict the output.

**What does this print?**

```python
records = {"ann": [90, 85]}
for name, index in [("ann", 5), ("bob", 0)]:
    try:
        print(records[name][index])
    except LookupError as e:
        print("lookup failed:", type(e).__name__)
```

<details><summary><b>Hint</b></summary>

`LookupError` is the parent of two exceptions you know. Which two?

</details>

<details><summary><b>Expected output</b></summary>

```text
lookup failed: IndexError
lookup failed: KeyError
```

</details>

<details><summary><b>Solution</b></summary>

`records["ann"][5]` raises `IndexError`; `records["bob"]` raises `KeyError`. Both are children of `LookupError`, so the one clause catches both. `type(e).__name__` shows which one actually happened. Catching a parent class catches all its children.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print([cls.__name__ for cls in KeyError.__mro__])
```

`KeyError.__mro__` lists a class and all its parents (NB14 explains it): `KeyError`, `LookupError`, `Exception`, `BaseException`, `object`. The full tree is in the docs under "Exception hierarchy", worth a bookmark.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB12-Q11 · L2 · Built-in Exceptions · Error

The code tries to protect itself. Which error does it raise?

**Which error does this raise?**

```python
text = "abc"
try:
    print(int(text))
except TypeError:
    print("that was not a number")
```

<details><summary><b>Hint</b></summary>

Is the problem the *type* of `"abc"`, or its *value*? Does the `except` match?

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError
```

</details>

<details><summary><b>Solution</b></summary>

`ValueError`: `int("abc")` gets a string (a fine type) with a bad value. The `except TypeError` clause doesn't match, so the error passes straight through it and crashes the program. An `except` only catches the classes it names (and their children).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
text = "abc"
try:
    print(int(text))
except ValueError:
    print("that was not a number")
```

When you're not sure which exception a call raises, try it in a scratch cell and read the last line of the traceback. Guessing the class is a common source of handlers that never run.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB12-Q12 · L2 · Custom Exceptions · Write

Define your own exception `OutOfStock` with the one-line form
`class OutOfStock(Exception): pass` (split over two lines). Write
`reserve(stock, item)`: if the item's count is `0`, raise
`OutOfStock(item)`; otherwise lower the count by one. Process every order:

```text
reserved mug
out of stock: lamp
out of stock: mug
```

**Starting code (already in the cell below):**

```python
stock = {"mug": 1, "lamp": 0}
orders = ["mug", "lamp", "mug"]
```

<details><summary><b>Hint</b></summary>

Inherit from `Exception`, and `pass` makes an empty class body. Catch it with `except OutOfStock as e:` like any built-in error.

</details>

<details><summary><b>Expected output</b></summary>

```text
reserved mug
out of stock: lamp
out of stock: mug
```

</details>

<details><summary><b>Solution</b></summary>

```python
class OutOfStock(Exception):
    pass

def reserve(stock, item):
    if stock[item] == 0:
        raise OutOfStock(item)
    stock[item] -= 1

for item in orders:
    try:
        reserve(stock, item)
        print("reserved", item)
    except OutOfStock as e:
        print("out of stock:", e)
```

`class OutOfStock(Exception): pass` creates a new exception type that behaves exactly like `Exception` but has its own name. Now the caller can catch *this* problem specifically, without accidentally catching a `KeyError` from a typo. The second `mug` order fails because the first one used the last unit.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class OutOfStockError(Exception):
    """Raised when an item has no units left."""

def reserve(stock, item):
    if stock[item] == 0:
        raise OutOfStockError(item)
    stock[item] -= 1

for item in orders:
    try:
        reserve(stock, item)
    except OutOfStockError as e:
        print("out of stock:", e)
    else:
        print("reserved", item)
```

PEP 8 says exception names should end in `Error` when they are errors. A docstring can replace `pass` as the class body, and it documents when the error is raised. Moving the success message to `else` keeps the `try` to the one risky call.

</details>

In [ ]:
stock = {"mug": 1, "lamp": 0}
orders = ["mug", "lamp", "mug"]

# your code here


### NB12-Q13 · L2 · Catching Too Broadly · Fix

This should print `2.5 0.0` (the price of tea, and `0.0` for anything not
on the menu). It prints `0.0 0.0`. Find the bug, and change the `except`
so a bug like this can never hide again.

**Buggy code (copied into the cell below):**

```python
prices = {"tea": 2.5, "cake": 3.0}

def price_of(item):
    try:
        return prices[itme]
    except:
        return 0.0

print(price_of("tea"), price_of("bread"))
```

<details><summary><b>Hint</b></summary>

Run `prices[itme]` on its own. What error do you get? Then ask: which error *should* this `except` be catching?

</details>

<details><summary><b>Expected output</b></summary>

```text
2.5 0.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
prices = {"tea": 2.5, "cake": 3.0}

def price_of(item):
    try:
        return prices[item]
    except KeyError:
        return 0.0

print(price_of("tea"), price_of("bread"))
```

`itme` is a typo, so every call raises `NameError`. The bare `except:` catches *everything*, including that `NameError`, and quietly returns `0.0`. The program never crashes, so the bug is invisible. Catching only `KeyError` (the error you expect from a missing dish) would have let the `NameError` crash loudly the first time you ran it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
prices = {"tea": 2.5, "cake": 3.0}

def price_of(item):
    return prices.get(item, 0.0)

print(price_of("tea"), price_of("bread"))
```

`dict.get()` with a default removes the need for `try` here. The bigger rule: always name the exception you expect. Linters flag bare `except:` (ruff `E722`) for exactly this reason.

</details>

In [ ]:
prices = {"tea": 2.5, "cake": 3.0}

def price_of(item):
    try:
        return prices[itme]
    except:
        return 0.0

print(price_of("tea"), price_of("bread"))


### NB12-Q14 · L2 · assert · Error

An `assert` guards the average. Which error does this raise?

**Which error does this raise?**

```python
def average(scores):
    assert len(scores) > 0, "no scores to average"
    return sum(scores) / len(scores)

print(average([80, 90]))
print(average([]))
```

<details><summary><b>Hint</b></summary>

`assert condition, message` does nothing when the condition is true. What does it do when it's false?

</details>

<details><summary><b>Expected output</b></summary>

```text
85.0
AssertionError
```

</details>

<details><summary><b>Solution</b></summary>

`85.0` is printed, then `average([])` raises `AssertionError`. `assert` is a quick self-check: "this should always be true here". But Python can be run with `-O` (optimise), which **removes every assert**. Then the empty list reaches the division and raises `ZeroDivisionError` instead. So never use `assert` to check input from users or other code; use `if` and `raise ValueError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def average(scores):
    if not scores:
        raise ValueError("no scores to average")
    return sum(scores) / len(scores)

print(average([80, 90]))
```

Seniors use `assert` for internal assumptions and in tests (pytest is built on it, NB28), and `raise` for validating arguments. A `ValueError` can't be switched off by a command-line flag.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L3 · Intermediate

### NB12-Q15 · L3 · else / finally · Predict

Both `try` and `finally` contain a `return`. Predict the output.

In Python 3.14 this code also shows a `SyntaxWarning`, which is a hint
about the answer. Predict only the printed value.

**What does this print?**

```python
def status():
    try:
        return "from try"
    finally:
        return "from finally"

print(status())
```

<details><summary><b>Hint</b></summary>

`finally` always runs, even after `return`. What happens if `finally` itself returns?

</details>

<details><summary><b>Expected output</b></summary>

```text
from finally
```

</details>

<details><summary><b>Solution</b></summary>

`from finally`. The `try` block's `return` prepares the value `"from try"`, but `finally` must run before the function really exits. Its own `return` replaces the pending value. It would also throw away an exception if one were on its way out (see Q29). Because this is almost always a bug, Python 3.14 shows `SyntaxWarning: 'return' in a 'finally' block` (PEP 765). The verifier for this course runs with warnings switched off, so only the printed value counts.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def status():
    try:
        result = "from try"
    finally:
        print("cleaning up")
    return result

print(status())
```

Seniors never put `return`, `break` or `continue` in `finally`. Clean-up goes in `finally`; the result is returned after the whole statement.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB12-Q16 · L3 · Re-raise · Write

Write `load_prices(texts)` that converts each text to a `float`. When one
fails, it should print `bad price: '4,50'` (the text in quotes) and then
**re-raise the same error** so the caller knows the import failed. The
caller catches it and prints `import stopped`.

```text
bad price: '4,50'
import stopped
```

**Starting code (already in the cell below):**

```python
texts = ["2.50", "4,50", "1.00"]
```

<details><summary><b>Hint</b></summary>

Inside an `except` block, a bare `raise` (with nothing after it) re-raises the exception you're handling.

</details>

<details><summary><b>Expected output</b></summary>

```text
bad price: '4,50'
import stopped
```

</details>

<details><summary><b>Solution</b></summary>

```python
def load_prices(texts):
    prices = []
    for text in texts:
        try:
            prices.append(float(text))
        except ValueError:
            print(f"bad price: {text!r}")
            raise
    return prices

try:
    load_prices(texts)
except ValueError:
    print("import stopped")
```

The function handles the part it knows about (which text was bad) and then passes the error on with a bare `raise`. The original exception, with its original traceback, travels up to the caller. Without the `raise`, the function would return a partial list and the caller would never know anything went wrong.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def load_prices(texts):
    prices = []
    for text in texts:
        try:
            prices.append(float(text))
        except ValueError as e:
            e.add_note(f"bad price: {text!r}")
            raise
    return prices

try:
    load_prices(texts)
except ValueError as e:
    print(*e.__notes__, sep="\n")
    print("import stopped")
```

A low-level function usually shouldn't print. `add_note()` (Q28) attaches the context to the exception itself, and the caller decides whether to print, log or ignore it. Notes also appear at the bottom of the traceback if nobody catches the error.

</details>

In [ ]:
texts = ["2.50", "4,50", "1.00"]

# your code here


### NB12-Q17 · L3 · raise ... from · Write

Write `get_setting(name)` that returns `settings[name]`. If the key is
missing, catch the `KeyError` and raise `ConfigError(f"missing setting:
{name}")` **from** it. Call `get_setting("port")`, catch the
`ConfigError`, and print its message and the class name of its cause:

```text
missing setting: port
caused by: KeyError
```

**Starting code (already in the cell below):**

```python
class ConfigError(Exception):
    pass

settings = {"host": "localhost"}
```

<details><summary><b>Hint</b></summary>

`raise NewError(...) from e` links the new error to the old one. The link is stored in the new error's `__cause__` attribute.

</details>

<details><summary><b>Expected output</b></summary>

```text
missing setting: port
caused by: KeyError
```

</details>

<details><summary><b>Solution</b></summary>

```python
def get_setting(name):
    try:
        return settings[name]
    except KeyError as e:
        raise ConfigError(f"missing setting: {name}") from e

try:
    get_setting("port")
except ConfigError as e:
    print(e)
    print("caused by:", type(e.__cause__).__name__)
```

This is *exception chaining*. Callers of `get_setting` shouldn't need to know that settings live in a dict (tomorrow it might be a file), so they get a `ConfigError` that speaks their language. `from e` keeps the original `KeyError` as `__cause__`, so nothing is lost when debugging: the traceback shows both, joined by "The above exception was the direct cause of the following exception".

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class ConfigError(LookupError):
    pass

def get_setting(name):
    try:
        return settings[name]
    except KeyError as e:
        raise ConfigError(f"missing setting: {name}") from e

try:
    get_setting("port")
except ConfigError as e:
    print(e)
    print("caused by:", type(e.__cause__).__name__)
```

Choosing the parent class is a design decision. Making `ConfigError` a kind of `LookupError` means older code that already catches `LookupError` keeps working, while new code can catch the precise `ConfigError`.

</details>

In [ ]:
class ConfigError(Exception):
    pass

settings = {"host": "localhost"}

# your code here


### NB12-Q18 · L3 · EAFP vs LBYL · Write

Write `get_port(config)` in **EAFP** style ("easier to ask forgiveness than
permission"): try to return `int(config["port"])`, and if the key is
missing or the value isn't a number, return `8080`. Print the port for
each config on one line:

```text
8000 8080 8080
```

**Starting code (already in the cell below):**

```python
configs = [{"port": "8000"}, {}, {"port": "eighty"}]
```

<details><summary><b>Hint</b></summary>

Two different errors can happen. One `except` clause can catch both if you give it a tuple: `except (KeyError, ValueError):`.

</details>

<details><summary><b>Expected output</b></summary>

```text
8000 8080 8080
```

</details>

<details><summary><b>Solution</b></summary>

```python
def get_port(config):
    try:
        return int(config["port"])
    except (KeyError, ValueError):
        return 8080

print(get_port(configs[0]), get_port(configs[1]), get_port(configs[2]))
```

**LBYL** ("look before you leap") checks everything first: is the key there, does the text look like a number? **EAFP** just tries and handles failure. EAFP is the usual Python style: the check and the action can't disagree (`int()` decides what's a number, not your own guess), and the normal path stays clean. A tuple after `except` catches any of the listed classes.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
DEFAULT_PORT = 8080

def get_port(config, default=DEFAULT_PORT):
    try:
        return int(config["port"])
    except (KeyError, ValueError):
        return default

print(*(get_port(config) for config in configs))
```

A named default that callers can override makes the function reusable. `print(*generator)` (NB09, NB10) prints every result separated by spaces, however many configs there are.

</details>

In [ ]:
configs = [{"port": "8000"}, {}, {"port": "eighty"}]

# your code here


### NB12-Q19 · L3 · EAFP vs LBYL · Refactor

This checks each order quantity before converting it (LBYL), with special
cases for negative numbers. Refactor it to EAFP with `try`/`except`/`else`.
Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
orders = ["3", "-2", "abc", "10"]
for text in orders:
    if text.isdigit():
        print(int(text) * 4.5)
    elif text.startswith("-") and text[1:].isdigit():
        print(int(text) * 4.5)
    else:
        print("invalid:", text)
```

<details><summary><b>Hint</b></summary>

Let `int()` decide what counts as a whole number. What should happen when it fails, and what when it succeeds?

</details>

<details><summary><b>Expected output</b></summary>

```text
13.5
-9.0
invalid: abc
45.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
orders = ["3", "-2", "abc", "10"]
for text in orders:
    try:
        qty = int(text)
    except ValueError:
        print("invalid:", text)
    else:
        print(qty * 4.5)
```

The LBYL version tries to predict what `int()` will accept, and it's still incomplete: `"+3"` and `" 7"` are valid for `int()` but rejected by these checks. The EAFP version asks `int()` itself, so it can never disagree with it, and the duplicated `int(text) * 4.5` line disappears.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
PRICE = 4.5
orders = ["3", "-2", "abc", "10"]
for text in orders:
    try:
        qty = int(text)
    except ValueError:
        print("invalid:", text)
        continue
    print(qty * PRICE)
```

`continue` in the handler is an alternative to `else`: the happy path stays at the normal indentation level, which reads well when there's more than one line of it. The magic number `4.5` gets a name.

</details>

In [ ]:
orders = ["3", "-2", "abc", "10"]
for text in orders:
    if text.isdigit():
        print(int(text) * 4.5)
    elif text.startswith("-") and text[1:].isdigit():
        print(int(text) * 4.5)
    else:
        print("invalid:", text)


### NB12-Q20 · L3 · __cause__ vs __context__ · Predict

A new error is raised *while handling* another one, without `from`.
Predict the output.

**What does this print?**

```python
try:
    try:
        {}["x"]
    except KeyError:
        int("y")
except ValueError as e:
    print(type(e).__name__)
    print(type(e.__context__).__name__)
    print(e.__cause__)
```

<details><summary><b>Hint</b></summary>

`__context__` is filled in automatically when an error happens inside an `except` block. `__cause__` is only set by `raise ... from`.

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError
KeyError
None
```

</details>

<details><summary><b>Solution</b></summary>

`{}["x"]` raises `KeyError`; its handler then raises `ValueError` from `int("y")`. The outer clause catches the `ValueError`. Python automatically records the `KeyError` it was handling as `__context__` (the traceback says "During handling of the above exception, another exception occurred"). `__cause__` is `None` because nobody wrote `from`. So: context means "happened while handling", cause means "deliberately raised because of".

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    try:
        {}["x"]
    except KeyError:
        raise ValueError("bad config") from None
except ValueError as e:
    print(e.__cause__, e.__suppress_context__)
```

`raise ... from None` tells Python the old error isn't interesting, which hides it from the traceback (`__suppress_context__` becomes `True`). Seniors use `from e` when the cause helps debugging and `from None` when it's pure noise.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB12-Q21 · L3 · e.args · Write

`check_row(row, line_no)` should raise `RowError("missing price", line_no)`
when `row` has no `"price"` key. Call it for row `{"name": "tea"}` on line
`3`, catch the error, print its `args`, then unpack them into a message:

```text
('missing price', 3)
line 3: missing price
```

**Starting code (already in the cell below):**

```python
class RowError(Exception):
    pass
```

<details><summary><b>Hint</b></summary>

Every argument you pass when creating an exception is stored, in order, in the tuple `e.args`. You can unpack it like any tuple.

</details>

<details><summary><b>Expected output</b></summary>

```text
('missing price', 3)
line 3: missing price
```

</details>

<details><summary><b>Solution</b></summary>

```python
def check_row(row, line_no):
    if "price" not in row:
        raise RowError("missing price", line_no)

try:
    check_row({"name": "tea"}, 3)
except RowError as e:
    print(e.args)
    message, line_no = e.args
    print(f"line {line_no}: {message}")
```

An exception can carry several values. They're stored in `e.args` as a tuple, so `message, line_no = e.args` unpacks them (NB05). With more than one argument, `print(e)` would show the whole tuple, which is why we build the message ourselves.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class RowError(Exception):
    def __init__(self, message, line_no):
        super().__init__(message, line_no)
        self.message = message
        self.line_no = line_no

def check_row(row, line_no):
    if "price" not in row:
        raise RowError("missing price", line_no)

try:
    check_row({"name": "tea"}, 3)
except RowError as e:
    print(e.args)
    print(f"line {e.line_no}: {e.message}")
```

Positions in `e.args` are easy to mix up. A full class with `__init__` (NB14) gives each value a name, `e.line_no` instead of `e.args[1]`. Passing the values to `super().__init__` keeps `e.args` working too.

</details>

In [ ]:
class RowError(Exception):
    pass

# your code here


### NB12-Q22 · L3 · Order of except Clauses · Fix

Asking for a product that isn't stocked should print `unknown product`,
but this prints `lookup problem`. Fix it.

**Buggy code (copied into the cell below):**

```python
stock = {"apple": 12}
try:
    print(stock["pear"])
except LookupError:
    print("lookup problem")
except KeyError:
    print("unknown product")
```

<details><summary><b>Hint</b></summary>

Python uses the **first** clause that matches. Is `KeyError` a kind of `LookupError`?

</details>

<details><summary><b>Expected output</b></summary>

```text
unknown product
```

</details>

<details><summary><b>Solution</b></summary>

```python
stock = {"apple": 12}
try:
    print(stock["pear"])
except KeyError:
    print("unknown product")
except LookupError:
    print("lookup problem")
```

`KeyError` is a child of `LookupError`, so the `except LookupError` clause matches first and the `KeyError` clause can **never** run. Always put the most specific class first and the general one after it, like sorting a post room: named people first, "everyone else" last.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
stock = {"apple": 12}
try:
    print(stock["pear"])
except KeyError:
    print("unknown product")
```

A dict lookup can only raise `KeyError`, so the general clause was dead code. Seniors catch exactly what the `try` block can raise. Pylint's `bad-except-order` check flags a child clause placed after its parent.

</details>

In [ ]:
stock = {"apple": 12}
try:
    print(stock["pear"])
except LookupError:
    print("lookup problem")
except KeyError:
    print("unknown product")


### NB12-Q23 · L3 · Custom Exceptions · Write

Build an exception family: `BankError(Exception)`, and two children
`InsufficientFunds(BankError)` and `AccountFrozen(BankError)`.

Write `withdraw(name, amount)`. It raises `AccountFrozen(name)` if the
account is in `frozen`, raises `InsufficientFunds(f"{name} has {balance},
needs {amount}")` if the balance is too low, and otherwise subtracts the
amount. Process every request with **one** `except BankError` clause:

```text
ok: ann now has 70
InsufficientFunds: ben has 50, needs 80
AccountFrozen: cara
ok: ann now has 0
```

**Starting code (already in the cell below):**

```python
accounts = {"ann": 100, "ben": 50, "cara": 75}
frozen = ["cara"]
requests = [("ann", 30), ("ben", 80), ("cara", 10), ("ann", 70)]
```

<details><summary><b>Hint</b></summary>

A child class is written `class Child(Parent): pass`. Catching `BankError` catches both children; `type(e).__name__` tells you which one.

</details>

<details><summary><b>Expected output</b></summary>

```text
ok: ann now has 70
InsufficientFunds: ben has 50, needs 80
AccountFrozen: cara
ok: ann now has 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
class BankError(Exception):
    pass

class InsufficientFunds(BankError):
    pass

class AccountFrozen(BankError):
    pass

def withdraw(name, amount):
    if name in frozen:
        raise AccountFrozen(name)
    balance = accounts[name]
    if amount > balance:
        raise InsufficientFunds(f"{name} has {balance}, needs {amount}")
    accounts[name] = balance - amount

for name, amount in requests:
    try:
        withdraw(name, amount)
    except BankError as e:
        print(f"{type(e).__name__}: {e}")
    else:
        print(f"ok: {name} now has {accounts[name]}")
```

A small hierarchy gives callers a choice: catch `BankError` to handle every banking problem in one place (as here), or catch `InsufficientFunds` alone to offer an overdraft. The checks happen **before** any money moves, so a failed withdrawal leaves the balance untouched.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class BankError(Exception):
    """Base class for every banking problem."""

class InsufficientFunds(BankError):
    """The balance is too low for the withdrawal."""

class AccountFrozen(BankError):
    """The account may not be used."""

def withdraw(name, amount):
    if name in frozen:
        raise AccountFrozen(name)
    balance = accounts[name]
    if amount > balance:
        raise InsufficientFunds(f"{name} has {balance}, needs {amount}")
    accounts[name] = balance - amount

for name, amount in requests:
    try:
        withdraw(name, amount)
    except BankError as e:
        print(f"{type(e).__name__}: {e}")
    else:
        print(f"ok: {name} now has {accounts[name]}")
```

Libraries such as `requests` work exactly like this: one base exception per package, with specific children. A docstring per class documents when each is raised. And catching `BankError` (not `Exception`) means a typo or an unknown account name still crashes loudly as a `KeyError`.

</details>

In [ ]:
accounts = {"ann": 100, "ben": 50, "cara": 75}
frozen = ["cara"]
requests = [("ann", 30), ("ben", 80), ("cara", 10), ("ann", 70)]

# your code here


### NB12-Q24 · L3 · traceback Module · Write

Call `load(["tea,3", "cake,x"])` and catch the `ValueError`. Use
`traceback.extract_tb(e.__traceback__)` to get the stack of calls the
error passed through, and print where it was raised and the whole path:

```text
ValueError raised in convert
path: <module> -> load -> parse -> convert
```

**Starting code (already in the cell below):**

```python
import traceback

def convert(text):
    return int(text)

def parse(line):
    return convert(line.split(",")[1])

def load(lines):
    numbers = []
    for line in lines:
        numbers.append(parse(line))
    return numbers
```

<details><summary><b>Hint</b></summary>

`extract_tb()` returns a list of frames, outermost first. Each frame has a `.name`: the function it was in. `<module>` is the top level of your cell.

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError raised in convert
path: <module> -> load -> parse -> convert
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    load(["tea,3", "cake,x"])
except ValueError as e:
    names = []
    for frame in traceback.extract_tb(e.__traceback__):
        names.append(frame.name)
    print(type(e).__name__, "raised in", names[-1])
    print("path:", " -> ".join(names))
```

Every exception carries its traceback in `e.__traceback__`: the chain of calls from where it was caught down to where it was raised. That's what Python prints when a program crashes. The `traceback` module turns it into data you can use, and `traceback.print_exc()` prints the full report without stopping the program. `int()` itself doesn't appear, because built-in functions written in C don't add frames.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    load(["tea,3", "cake,x"])
except ValueError as e:
    names = [frame.name for frame in traceback.extract_tb(e.__traceback__)]
    print(type(e).__name__, "raised in", names[-1])
    print("path:", " -> ".join(names))
```

In real programs seniors rarely read frames by hand: `logging.exception("import failed")` (NB28) writes the message *and* the full traceback to the log. `extract_tb` is for tools that need the data, such as error reporters.

</details>

In [ ]:
import traceback

def convert(text):
    return int(text)

def parse(line):
    return convert(line.split(",")[1])

def load(lines):
    numbers = []
    for line in lines:
        numbers.append(parse(line))
    return numbers

# your code here


### NB12-Q25 · L3 · Several except Clauses · Write

Convert each value with `int()`. Some fail with `ValueError`, some with
`TypeError`. Catch both in **one** clause and print which one happened:

```text
12
skipped ValueError
skipped TypeError
7
```

**Starting code (already in the cell below):**

```python
values = ["12", "abc", None, 7.9]
```

<details><summary><b>Hint</b></summary>

`except (ValueError, TypeError) as e:` catches either. `type(e).__name__` gives the class name.

</details>

<details><summary><b>Expected output</b></summary>

```text
12
skipped ValueError
skipped TypeError
7
```

</details>

<details><summary><b>Solution</b></summary>

```python
for value in values:
    try:
        print(int(value))
    except (ValueError, TypeError) as e:
        print("skipped", type(e).__name__)
```

`int("abc")` is a `ValueError` (a string, but not a number), while `int(None)` is a `TypeError` (`None` can't be converted at all). `int(7.9)` works and truncates to `7`. A tuple of classes catches any of them, which is cleaner than two clauses with the same body.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for value in values:
    try:
        number = int(value)
    except (ValueError, TypeError) as e:
        print("skipped", type(e).__name__)
    else:
        print(number)
```

The `else` keeps `print` out of the `try`. From Python 3.14 the brackets can be dropped when there's no `as` (Q26), but with `as` they're still required, and code that must run on older versions needs them anyway.

</details>

In [ ]:
values = ["12", "abc", None, 7.9]

# your code here


### NB12-Q26 · L3 · except Without Brackets · Predict [3.14+]

Python 3.14 lets you list several exceptions without brackets, as long as
there's no `as`. Predict the output.

**What does this print?**

```python
def to_number(value):
    try:
        return float(value)
    except ValueError, TypeError:
        return 0.0

print(to_number("2.5"), to_number("two"), to_number(None))
```

<details><summary><b>Hint</b></summary>

`except ValueError, TypeError:` means exactly the same as `except (ValueError, TypeError):`.

</details>

<details><summary><b>Expected output</b></summary>

```text
2.5 0.0 0.0
```

</details>

<details><summary><b>Solution</b></summary>

`float("2.5")` works. `float("two")` raises `ValueError` and `float(None)` raises `TypeError`; the clause catches both and returns `0.0`. Before 3.14 this line was a `SyntaxError`, because Python 2 used `except ValueError, e:` to mean something completely different. PEP 758 allowed it again once Python 2 was long gone. Adding `as` still needs the brackets.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def to_number(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return 0.0

print(to_number("2.5"), to_number("two"), to_number(None))
```

Seniors keep the brackets in code that must also run on Python 3.13 or older, and in any `except ... as e`. The bracket-free form is fine in 3.14-only projects.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L4 · Somewhat Difficult

### NB12-Q27 · L4 · Retry · Write

`fetch_rates()` fails twice with `ConnectionError`, then works. Call it up
to 3 times. Print each failure, and stop as soon as it works. If all 3
attempts fail, print `giving up`.

```text
attempt 1 failed: server busy
attempt 2 failed: server busy
got rates on attempt 3: {'EUR': 0.92}
```

**Starting code (already in the cell below):**

```python
attempts_made = 0

def fetch_rates():
    global attempts_made
    attempts_made += 1
    if attempts_made < 3:
        raise ConnectionError("server busy")
    return {"EUR": 0.92}
```

<details><summary><b>Hint</b></summary>

A `for` loop over `range(1, 4)` with `try`/`except`/`else` inside. `break` when it works, and the loop's own `else` (NB04) runs only if it never broke.

</details>

<details><summary><b>Expected output</b></summary>

```text
attempt 1 failed: server busy
attempt 2 failed: server busy
got rates on attempt 3: {'EUR': 0.92}
```

</details>

<details><summary><b>Solution</b></summary>

```python
for attempt in range(1, 4):
    try:
        rates = fetch_rates()
    except ConnectionError as e:
        print(f"attempt {attempt} failed: {e}")
    else:
        print(f"got rates on attempt {attempt}: {rates}")
        break
else:
    print("giving up")
```

Each failure is caught and the loop tries again. The `try`'s `else` runs on success and `break`s out. The *loop's* `else` only runs if the loop finished without `break`, meaning every attempt failed. Catching only `ConnectionError` matters: a `TypeError` from a bug shouldn't be retried, it should crash.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def retry(func, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            result = func()
        except ConnectionError as e:
            print(f"attempt {attempt} failed: {e}")
            if attempt == attempts:
                raise
        else:
            print(f"got rates on attempt {attempt}: {result}")
            return result

try:
    retry(fetch_rates)
except ConnectionError:
    print("giving up")
```

Seniors put retry logic in a reusable function (later a decorator, NB23) that re-raises the last error instead of hiding it. Real retry code also waits longer between attempts (*exponential backoff*) so a struggling server gets time to recover.

</details>

In [ ]:
attempts_made = 0

def fetch_rates():
    global attempts_made
    attempts_made += 1
    if attempts_made < 3:
        raise ConnectionError("server busy")
    return {"EUR": 0.92}

# your code here


### NB12-Q28 · L4 · add_note · Write

Write `parse_row(text, line_no)` that splits `"name,price"` and returns the
name and `float(price)`. If the conversion fails, add a note
`line <n>: '<text>'` to the exception with `add_note()` and re-raise it.
Parse every row, printing each one, and stop at the first error. Catch it
at the top and print the error's class name and its notes:

```text
tea 2.5
ValueError
note: line 2: 'cake,abc'
```

**Starting code (already in the cell below):**

```python
rows = ["tea,2.50", "cake,abc", "jam,3.10"]
```

<details><summary><b>Hint</b></summary>

`e.add_note("...")` attaches a string to the exception; `raise` sends it on. The notes are kept in the list `e.__notes__`. `enumerate(rows, start=1)` gives line numbers from 1.

</details>

<details><summary><b>Expected output</b></summary>

```text
tea 2.5
ValueError
note: line 2: 'cake,abc'
```

</details>

<details><summary><b>Solution</b></summary>

```python
def parse_row(text, line_no):
    name, price = text.split(",")
    try:
        return name, float(price)
    except ValueError as e:
        e.add_note(f"line {line_no}: {text!r}")
        raise

try:
    for line_no, text in enumerate(rows, start=1):
        name, price = parse_row(text, line_no)
        print(name, price)
except ValueError as e:
    print(type(e).__name__)
    for note in e.__notes__:
        print("note:", note)
```

`add_note()` (Python 3.11+) adds context to an exception without changing its type or message, so callers that catch `ValueError` still work. The notes are stored in `e.__notes__` and are printed under the error message in a traceback. It answers the classic debugging question: *which* row was bad? `jam` is never reached, because the error stops the loop.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def parse_row(text, line_no):
    name, price = text.split(",")
    try:
        return name, float(price)
    except ValueError as e:
        e.add_note(f"line {line_no}: {text!r}")
        raise

try:
    for line_no, text in enumerate(rows, start=1):
        print(*parse_row(text, line_no))
except ValueError as e:
    print(type(e).__name__)
    print(*(f"note: {note}" for note in e.__notes__), sep="\n")
```

Before notes existed, people wrapped errors in new exceptions just to add a line number, losing the original type. Notes keep the type and add the context, which makes them a good fit for parsers and batch jobs. Use `getattr(e, "__notes__", [])` when an error might have none.

</details>

In [ ]:
rows = ["tea,2.50", "cake,abc", "jam,3.10"]

# your code here


### NB12-Q29 · L4 · else / finally · Fix

`save("")` should fail with `save failed: nothing to save`, but the error
disappears and it prints `done`. Fix it so the output is:

```text
saving order 17
closing file
done
closing file
save failed: nothing to save
```

**Buggy code (copied into the cell below):**

```python
def save(record):
    try:
        if not record:
            raise ValueError("nothing to save")
        print("saving", record)
    finally:
        print("closing file")
        return "done"

for record in ["order 17", ""]:
    try:
        print(save(record))
    except ValueError as e:
        print("save failed:", e)
```

<details><summary><b>Hint</b></summary>

What does a `return` inside `finally` do to an exception that is on its way out (Q15)?

</details>

<details><summary><b>Expected output</b></summary>

```text
saving order 17
closing file
done
closing file
save failed: nothing to save
```

</details>

<details><summary><b>Solution</b></summary>

```python
def save(record):
    try:
        if not record:
            raise ValueError("nothing to save")
        print("saving", record)
    finally:
        print("closing file")
    return "done"

for record in ["order 17", ""]:
    try:
        print(save(record))
    except ValueError as e:
        print("save failed:", e)
```

A `return` in `finally` doesn't only replace a return value (Q15): it also **cancels an exception** that is passing through. The `ValueError` is silently thrown away and the caller sees `"done"`. Moving `return` after the `try` statement means it only runs when no error escaped, while `closing file` still prints both times. Python 3.14 warns about the buggy version with a `SyntaxWarning` (PEP 765).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def save(record):
    try:
        if not record:
            raise ValueError("nothing to save")
        print("saving", record)
    finally:
        print("closing file")
    return "done"

for record in ["order 17", ""]:
    try:
        result = save(record)
    except ValueError as e:
        print("save failed:", e)
    else:
        print(result)
```

The fix is the same. The senior habit: `finally` is for clean-up only, never for control flow (`return`, `break`, `continue`). For files, `with open(...)` (NB13) closes them for you, so there's no `finally` to get wrong.

</details>

In [ ]:
def save(record):
    try:
        if not record:
            raise ValueError("nothing to save")
        print("saving", record)
    finally:
        print("closing file")
        return "done"

for record in ["order 17", ""]:
    try:
        print(save(record))
    except ValueError as e:
        print("save failed:", e)


### NB12-Q30 · L4 · ExceptionGroup · Write

`submit_order()` reports **every** problem at once by raising an
`ExceptionGroup`. Handle it with `except*`: print each `ValueError` and
each `TypeError` inside the group:

```text
value problem: quantity must be positive
value problem: email is missing an @
type problem: price must be a number
```

**Starting code (already in the cell below):**

```python
def submit_order():
    raise ExceptionGroup("order rejected", [
        ValueError("quantity must be positive"),
        TypeError("price must be a number"),
        ValueError("email is missing an @"),
    ])
```

<details><summary><b>Hint</b></summary>

`except* ValueError as group:` receives a smaller group holding only the `ValueError`s. Loop over `group.exceptions`.

</details>

<details><summary><b>Expected output</b></summary>

```text
value problem: quantity must be positive
value problem: email is missing an @
type problem: price must be a number
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    submit_order()
except* ValueError as group:
    for e in group.exceptions:
        print("value problem:", e)
except* TypeError as group:
    for e in group.exceptions:
        print("type problem:", e)
```

An `ExceptionGroup` (Python 3.11+) carries several exceptions at once. Unlike `except`, **each** `except*` clause runs for its own share of the group: the first gets both `ValueError`s, the second gets the `TypeError`. `group.exceptions` is the tuple of errors inside the sub-group.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    submit_order()
except* ValueError as group:
    print(*(f"value problem: {e}" for e in group.exceptions), sep="\n")
except* TypeError as group:
    print(*(f"type problem: {e}" for e in group.exceptions), sep="\n")
```

Exception groups exist for situations where several things fail independently: validating a form, or running tasks concurrently (`asyncio.TaskGroup`, NB27). For one error at a time, plain `except` is still the right tool.

</details>

In [ ]:
def submit_order():
    raise ExceptionGroup("order rejected", [
        ValueError("quantity must be positive"),
        TypeError("price must be a number"),
        ValueError("email is missing an @"),
    ])

# your code here


### NB12-Q31 · L4 · ExceptionGroup · Error

The group holds a `ValueError` and a `KeyError`, but only `ValueError` is
handled. Predict the printed line, then the error.

**Which error does this raise?**

```python
try:
    raise ExceptionGroup("import failed", [ValueError("bad price"), KeyError("sku")])
except* ValueError as group:
    print("fixed", len(group.exceptions), "value problem")
```

<details><summary><b>Hint</b></summary>

`except*` handles its part of the group. What happens to the part nobody handled?

</details>

<details><summary><b>Expected output</b></summary>

```text
fixed 1 value problem
ExceptionGroup
```

</details>

<details><summary><b>Solution</b></summary>

The `except*` clause runs and prints `fixed 1 value problem`. The `KeyError` has no handler, so Python re-raises it, still wrapped in an `ExceptionGroup`. The error you see is `ExceptionGroup`, not `KeyError`. Nothing in a group is ever silently dropped: every exception is either handled or raised again.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    raise ExceptionGroup("import failed", [ValueError("bad price"), KeyError("sku")])
except* ValueError as group:
    print("fixed", len(group.exceptions), "value problem")
except* KeyError as group:
    print("missing", len(group.exceptions), "field")
```

When you call code that raises groups, handle every type it documents, or let the rest propagate on purpose. A plain `except ExceptionGroup` would also catch it, but then you'd have to split the group yourself.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB12-Q32 · L4 · Several except Clauses · Refactor

This nests one `try` inside another and repeats the same handlers twice.
Refactor it into a single `try` statement. Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def order_total(order):
    try:
        qty = int(order["qty"])
        try:
            price = float(order["price"])
            return qty * price
        except KeyError:
            return "missing field"
        except ValueError:
            return "not a number"
    except KeyError:
        return "missing field"
    except ValueError:
        return "not a number"

print(order_total({"qty": "3", "price": "2.5"}))
print(order_total({"qty": "3"}))
print(order_total({"qty": "three", "price": "2.5"}))
print(order_total({"qty": "3", "price": "cheap"}))
```

<details><summary><b>Hint</b></summary>

Both conversions can raise the same two errors, and both get the same message. One `try` can hold both lines.

</details>

<details><summary><b>Expected output</b></summary>

```text
7.5
missing field
not a number
not a number
```

</details>

<details><summary><b>Solution</b></summary>

```python
def order_total(order):
    try:
        qty = int(order["qty"])
        price = float(order["price"])
        return qty * price
    except KeyError:
        return "missing field"
    except ValueError:
        return "not a number"

print(order_total({"qty": "3", "price": "2.5"}))
print(order_total({"qty": "3"}))
print(order_total({"qty": "three", "price": "2.5"}))
print(order_total({"qty": "3", "price": "cheap"}))
```

When several steps fail in the same ways and need the same handling, one `try` block with one set of clauses covers them all. Nested `try` statements are only worth it when an inner failure needs *different* handling from an outer one.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def order_total(order):
    try:
        qty = int(order["qty"])
        price = float(order["price"])
    except KeyError:
        return "missing field"
    except ValueError:
        return "not a number"
    return qty * price

print(order_total({"qty": "3", "price": "2.5"}))
print(order_total({"qty": "3"}))
print(order_total({"qty": "three", "price": "2.5"}))
print(order_total({"qty": "3", "price": "cheap"}))
```

The calculation moves out of the `try`, which now holds only the lines that can fail. A senior would also question returning error *strings* from a function that normally returns a number: callers have to check the type. Raising a custom exception is usually the better design.

</details>

In [ ]:
def order_total(order):
    try:
        qty = int(order["qty"])
        try:
            price = float(order["price"])
            return qty * price
        except KeyError:
            return "missing field"
        except ValueError:
            return "not a number"
    except KeyError:
        return "missing field"
    except ValueError:
        return "not a number"

print(order_total({"qty": "3", "price": "2.5"}))
print(order_total({"qty": "3"}))
print(order_total({"qty": "three", "price": "2.5"}))
print(order_total({"qty": "3", "price": "cheap"}))


### NB12-Q33 · L4 · Exception Hierarchy · Predict

`KeyboardInterrupt` is what Python raises when you press Ctrl+C (or the
Jupyter stop button). Predict the output.

**What does this print?**

```python
try:
    try:
        raise KeyboardInterrupt
    except Exception:
        print("caught by except Exception")
except:
    print("caught by bare except")
```

<details><summary><b>Hint</b></summary>

Is `KeyboardInterrupt` a child of `Exception`? Look up where it sits in the family tree: it hangs off `BaseException` directly.

</details>

<details><summary><b>Expected output</b></summary>

```text
caught by bare except
```

</details>

<details><summary><b>Solution</b></summary>

`caught by bare except`. `KeyboardInterrupt` (like `SystemExit`) is a child of `BaseException`, **not** of `Exception`, on purpose: `except Exception` lets it through so you can still stop a program. A bare `except:` catches *everything*, including Ctrl+C, so a loop with a bare `except` inside can become impossible to interrupt.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    try:
        raise KeyboardInterrupt
    except Exception:
        print("caught by except Exception")
except KeyboardInterrupt:
    print("stopped by the user")
```

Seniors never write a bare `except:`. The broadest catch they write is `except Exception`, usually at the very top of a program to log the error. When they really mean Ctrl+C, they name `KeyboardInterrupt`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L5 · Hard

### NB12-Q34 · L5 · ExceptionGroup · Write

Write `validate(config)` that checks **all** of these rules and collects
every problem, instead of stopping at the first:

- `"host"` must be present: otherwise collect `KeyError("host")`
- `"port"` must convert with `int()`: otherwise collect
  `ValueError("port must be a whole number")`
- `"workers"` must be at least `1`: otherwise collect
  `ValueError("workers must be at least 1")`
- `"mode"` must be `"dev"` or `"prod"`: otherwise collect
  `ValueError("mode must be dev or prod")`

If there are problems, raise them together as
`ExceptionGroup("invalid config", problems)`; otherwise print `config ok`.
Validate both configs and handle the group with `except*`:

```text
config ok
missing: host
bad value: port must be a whole number
bad value: workers must be at least 1
bad value: mode must be dev or prod
```

**Starting code (already in the cell below):**

```python
good = {"host": "shop.local", "port": "8000", "workers": "4", "mode": "prod"}
bad = {"port": "80a", "workers": "0", "mode": "fast"}
```

<details><summary><b>Hint</b></summary>

Build a list `problems` and append exception *objects* to it (don't raise them yet). For printing a `KeyError`, use `e.args[0]`: `str()` of a `KeyError` adds quotes.

</details>

<details><summary><b>Expected output</b></summary>

```text
config ok
missing: host
bad value: port must be a whole number
bad value: workers must be at least 1
bad value: mode must be dev or prod
```

</details>

<details><summary><b>Solution</b></summary>

```python
def validate(config):
    problems = []
    if "host" not in config:
        problems.append(KeyError("host"))
    try:
        int(config["port"])
    except ValueError:
        problems.append(ValueError("port must be a whole number"))
    if int(config["workers"]) < 1:
        problems.append(ValueError("workers must be at least 1"))
    if config["mode"] not in ("dev", "prod"):
        problems.append(ValueError("mode must be dev or prod"))
    if problems:
        raise ExceptionGroup("invalid config", problems)
    print("config ok")

for config in [good, bad]:
    try:
        validate(config)
    except* KeyError as group:
        for e in group.exceptions:
            print("missing:", e.args[0])
    except* ValueError as group:
        for e in group.exceptions:
            print("bad value:", e)
```

Raising at the first problem would make the user fix one thing, run again, fix the next, and so on. Collecting exception objects in a list and raising them together reports everything at once. Two traps: an exception object does nothing until it's raised, so appending it is safe; and `str(KeyError("host"))` is `'host'` *with* quotes (a quirk of `KeyError`), which is why the handler prints `e.args[0]`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def is_int(text):
    try:
        int(text)
    except ValueError:
        return False
    return True

RULES = [
    ("port", is_int, "port must be a whole number"),
    ("workers", lambda text: is_int(text) and int(text) >= 1,
     "workers must be at least 1"),
    ("mode", lambda text: text in ("dev", "prod"), "mode must be dev or prod"),
]

def validate(config):
    problems = [] if "host" in config else [KeyError("host")]
    problems += [ValueError(message) for key, check, message in RULES
                 if not check(config.get(key, ""))]
    if problems:
        raise ExceptionGroup("invalid config", problems)
    print("config ok")

for config in [good, bad]:
    try:
        validate(config)
    except* KeyError as group:
        for e in group.exceptions:
            print("missing:", e.args[0])
    except* ValueError as group:
        for e in group.exceptions:
            print("bad value:", e)
```

A table of rules (NB09's dispatch idea) makes adding a rule a one-line change. `config.get(key, "")` means a *missing* port is also reported instead of crashing with a `KeyError` mid-validation, an edge the beginner version doesn't cover. Libraries like Pydantic do this kind of validation for real projects.

</details>

In [ ]:
good = {"host": "shop.local", "port": "8000", "workers": "4", "mode": "prod"}
bad = {"port": "80a", "workers": "0", "mode": "fast"}

# your code here


### NB12-Q35 · L5 · raise ... from · Write

Write `transfer(src, dst, amount)` for the `accounts` dict:

1. If `src` has less than `amount`, raise
   `InsufficientFunds(f"{src} has {balance}, needs {amount}")`.
2. Take the money out of `src`.
3. Add it to `dst`. If `dst` doesn't exist (`KeyError`), **put the money
   back** into `src`, then raise `UnknownAccount(dst)` from the `KeyError`.

Run every transfer, then print the balances:

```text
ok: ann -> ben 30
failed: InsufficientFunds: ben has 50, needs 500
failed: UnknownAccount: zoe
balances: {'ann': 70, 'ben': 50}
```

**Starting code (already in the cell below):**

```python
class InsufficientFunds(Exception):
    pass

class UnknownAccount(Exception):
    pass

accounts = {"ann": 100, "ben": 20}
transfers = [("ann", "ben", 30), ("ben", "ann", 500), ("ann", "zoe", 50)]
```

<details><summary><b>Hint</b></summary>

The rollback goes in the `except KeyError as e:` block, before the `raise ... from e`. In the loop, catch both custom exceptions with a tuple.

</details>

<details><summary><b>Expected output</b></summary>

```text
ok: ann -> ben 30
failed: InsufficientFunds: ben has 50, needs 500
failed: UnknownAccount: zoe
balances: {'ann': 70, 'ben': 50}
```

</details>

<details><summary><b>Solution</b></summary>

```python
def transfer(src, dst, amount):
    balance = accounts[src]
    if balance < amount:
        raise InsufficientFunds(f"{src} has {balance}, needs {amount}")
    accounts[src] -= amount
    try:
        accounts[dst] += amount
    except KeyError as e:
        accounts[src] += amount
        raise UnknownAccount(dst) from e

for src, dst, amount in transfers:
    try:
        transfer(src, dst, amount)
    except (InsufficientFunds, UnknownAccount) as e:
        print(f"failed: {type(e).__name__}: {e}")
    else:
        print(f"ok: {src} -> {dst} {amount}")
print("balances:", accounts)
```

The trap is the third transfer: the money has already left `ann` when the deposit fails. Without the rollback, `ann` would end with `20` and 50 units would vanish. Catching the `KeyError`, undoing the first step, then raising a clearer error keeps the books balanced. `from e` keeps the original `KeyError` as the cause, and catching only the two custom exceptions means real bugs still crash.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def transfer(src, dst, amount):
    if dst not in accounts:
        raise UnknownAccount(dst)
    balance = accounts[src]
    if balance < amount:
        raise InsufficientFunds(f"{src} has {balance}, needs {amount}")
    accounts[src] -= amount
    accounts[dst] += amount

for src, dst, amount in transfers:
    try:
        transfer(src, dst, amount)
    except (InsufficientFunds, UnknownAccount) as e:
        print(f"failed: {type(e).__name__}: {e}")
    else:
        print(f"ok: {src} -> {dst} {amount}")
print("balances:", accounts)
```

Seniors validate **everything before changing anything**, so there's nothing to roll back. When a rollback can't be avoided (two databases, a payment API), they use transactions: `sqlite3` (NB31) commits both steps or neither.

</details>

In [ ]:
class InsufficientFunds(Exception):
    pass

class UnknownAccount(Exception):
    pass

accounts = {"ann": 100, "ben": 20}
transfers = [("ann", "ben", 30), ("ben", "ann", 500), ("ann", "zoe", 50)]

# your code here


### NB12-Q36 · L5 · else / finally · Predict

`else`, `finally` and an error that happens in an unexpected place.
Predict the output.

**What does this print?**

```python
def run(text):
    try:
        n = int(text)
    except ValueError:
        print("bad number")
    else:
        print("got", n)
        print(10 / n)
    finally:
        print("done", text)

for text in ["5", "x"]:
    run(text)
try:
    run("0")
except ZeroDivisionError:
    print("outer caught it")
```

<details><summary><b>Hint</b></summary>

The `except` clauses only protect the `try` block, not the `else` block. And `finally` runs before an escaping error leaves the function.

</details>

<details><summary><b>Expected output</b></summary>

```text
got 5
2.0
done 5
bad number
done x
got 0
done 0
outer caught it
```

</details>

<details><summary><b>Solution</b></summary>

`"5"`: `got 5`, `2.0`, `done 5`. `"x"`: `bad number`, `done x`. For `"0"`, `int()` works, so `else` runs: `got 0`, then `10 / 0` raises `ZeroDivisionError` **inside `else`**. The `except ValueError` doesn't protect `else` (and wouldn't match anyway), so the error heads out of the function. On its way, `finally` prints `done 0`. Then the outer handler prints `outer caught it`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def run(text):
    try:
        n = int(text)
        share = 10 / n
    except ValueError:
        print("bad number")
    except ZeroDivisionError:
        print("cannot divide by zero")
    else:
        print("got", n)
        print(share)
    finally:
        print("done", text)

for text in ["5", "x", "0"]:
    run(text)
```

That's the point of `else`: code there is deliberately *not* protected. If an error in that code should be handled here too, move the risky line into `try` and give it its own clause, as the senior version does.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## Mini-project

Pull together everything from this notebook.

## NB12-P · L5 · Mini-project: Robust Record Parser · Write

`lines` is a small CSV export (the first line is the header). Parse every
data line into a record, and report **every** bad line with its line
number instead of stopping. Then print the good records and a total.

Define `RecordError(Exception)` with two children, `MissingField` and
`BadNumber`. `parse_record(line)` must raise:

- `MissingField(f"expected 4 fields, got {n}")` if a line doesn't have 4
  comma-separated fields
- `MissingField("price is empty")` if the price is blank
- `BadNumber(f"price {text!r} is not a number")` (from the `ValueError`)
  if `float()` fails
- `BadNumber("price must not be negative")` for a negative price
- `BadNumber(f"qty {text!r} is not a whole number")` (from the
  `ValueError`) if `int()` fails

Line numbers count the header as line 1. Money has 2 decimal places.

```text
line 3: MissingField: price is empty
line 4: BadNumber: qty 'two' is not a whole number
line 5: MissingField: expected 4 fields, got 3
line 7: BadNumber: price must not be negative
Parsed 2 of 6 records
Mug x3 = 22.50
Candle x4 = 20.00
Total: 42.50
```

**Starting code (already in the cell below):**

```python
lines = [
    "sku,name,price,qty",
    "A100,Mug,7.50,3",
    "A101,Lamp,,1",
    "A102,Plant,12.00,two",
    "A103,Poster,4.25",
    "A104,Candle,5.00,4",
    "A105,Rug,-20.00,1",
]
```

<details><summary><b>Hint</b></summary>

Write `parse_record` first and test it on one line at a time. In the main loop, use `enumerate(lines[1:], start=2)`, catch `RecordError` (the parent catches both children), and collect the good records in the `else` block.

</details>

<details><summary><b>Expected output</b></summary>

```text
line 3: MissingField: price is empty
line 4: BadNumber: qty 'two' is not a whole number
line 5: MissingField: expected 4 fields, got 3
line 7: BadNumber: price must not be negative
Parsed 2 of 6 records
Mug x3 = 22.50
Candle x4 = 20.00
Total: 42.50
```

</details>

<details><summary><b>Solution</b></summary>

```python
class RecordError(Exception):
    pass

class MissingField(RecordError):
    pass

class BadNumber(RecordError):
    pass

def parse_record(line):
    fields = line.split(",")
    if len(fields) != 4:
        raise MissingField(f"expected 4 fields, got {len(fields)}")
    sku, name, price_text, qty_text = fields
    if price_text == "":
        raise MissingField("price is empty")
    try:
        price = float(price_text)
    except ValueError as e:
        raise BadNumber(f"price {price_text!r} is not a number") from e
    if price < 0:
        raise BadNumber("price must not be negative")
    try:
        qty = int(qty_text)
    except ValueError as e:
        raise BadNumber(f"qty {qty_text!r} is not a whole number") from e
    return {"sku": sku, "name": name, "price": price, "qty": qty}

records = []
for line_no, line in enumerate(lines[1:], start=2):
    try:
        record = parse_record(line)
    except RecordError as e:
        print(f"line {line_no}: {type(e).__name__}: {e}")
    else:
        records.append(record)

print(f"Parsed {len(records)} of {len(lines) - 1} records")
total = 0
for record in records:
    line_total = record["price"] * record["qty"]
    total += line_total
    print(f"{record['name']} x{record['qty']} = {line_total:.2f}")
print(f"Total: {total:.2f}")
```

`parse_record` only *detects* problems and raises a precise exception; the loop *decides* what to do (report and carry on). Because the `try` is inside the loop, one bad line never stops the rest. The hierarchy lets the loop catch `RecordError` once while the report still names the exact child class. `raise ... from e` keeps the original `ValueError` as the cause for debugging, and anything that is *not* a `RecordError` (a real bug) still crashes loudly. The order of the checks matters too: the field count is checked before unpacking, and the empty price before `float()`, so each line gets the most helpful message.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import csv
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation

class RecordError(Exception):
    """A line of the export could not be turned into a record."""

class MissingField(RecordError):
    """A required field is absent or empty."""

class BadNumber(RecordError):
    """A numeric field holds something that isn't a valid number."""

@dataclass
class Record:
    sku: str
    name: str
    price: Decimal
    qty: int

    @property
    def total(self):
        return self.price * self.qty

def parse_record(fields):
    if len(fields) != 4:
        raise MissingField(f"expected 4 fields, got {len(fields)}")
    sku, name, price_text, qty_text = fields
    if not price_text:
        raise MissingField("price is empty")
    try:
        price = Decimal(price_text)
    except InvalidOperation as e:
        raise BadNumber(f"price {price_text!r} is not a number") from e
    if price < 0:
        raise BadNumber("price must not be negative")
    try:
        qty = int(qty_text)
    except ValueError as e:
        raise BadNumber(f"qty {qty_text!r} is not a whole number") from e
    return Record(sku, name, price, qty)

reader = csv.reader(lines)
next(reader)
records = []
for line_no, fields in enumerate(reader, start=2):
    try:
        records.append(parse_record(fields))
    except RecordError as e:
        print(f"line {line_no}: {type(e).__name__}: {e}")

print(f"Parsed {len(records)} of {len(lines) - 1} records")
for record in records:
    print(f"{record.name} x{record.qty} = {record.total:.2f}")
print(f"Total: {sum(record.total for record in records):.2f}")
```

`csv.reader` (NB13) handles quoted fields with commas inside, which `split(",")` gets wrong. `Decimal` (NB16) keeps money exact, and a `dataclass` (NB21) gives records named, typed fields instead of dict keys that can be misspelled. The exception design is the same as the beginner's: precise children under one parent, chained to the low-level cause.

</details>

In [ ]:
lines = [
    "sku,name,price,qty",
    "A100,Mug,7.50,3",
    "A101,Lamp,,1",
    "A102,Plant,12.00,two",
    "A103,Poster,4.25",
    "A104,Candle,5.00,4",
    "A105,Rug,-20.00,1",
]

# your code here


---

## Done

Next up: **NB13 · Modules Files JSON CSV** in `intermediate/`.